# 12 · Why Does Fidelity Fall with Dimension? A Look at Kues et al. 2017

**Question:** in Kues et al., *Nature* 546, 622 (2017), the tomographic fidelity of the frequency-bin entangled state falls from 0.885 (D = 2) to 0.809 (D = 3) to 0.766 (D = 4), while the two-photon interference visibility stays at about 86 %. Notebook 09 showed that a simple white-noise model cannot reproduce this. Here, competing explanations are written as density-matrix models, fitted to the three fidelities, and tested against other evidence.

**From the paper (accepted manuscript):** tomography uses maximum-likelihood reconstruction; visibilities are raw (no background subtraction); the authors name unequal mode amplitudes from SFWM and the 14.5 dB loss of the control setup (which lowers the coincidence-to-accidental ratio) as limitations. D = 2 and 3 mix bins with ±200 GHz sidebands at 11 % efficiency; D = 4 uses ±100 and ±300 GHz sidebands at 4 %.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

In [2]:
from scipy.optimize import least_squares
F_MEAS = {2: 0.885, 3: 0.809, 4: 0.766}
DIMS = [2, 3, 4]

def ideal(D):
    return sum(qt.tensor(qt.basis(D, k), qt.basis(D, k)) for k in range(D)).unit()

def state(D, p, r=1.0, c=1.0):
    # p: weight of the coherent part (the rest is white noise)
    # r: population ratio between neighbouring bins (1 = equal brightness)
    # c: coherence kept per bin of separation (1 = no dephasing)
    pops = r ** np.arange(D)
    pops = pops / pops.sum()
    rho = np.zeros((D * D, D * D), dtype=complex)
    for j in range(D):
        for k in range(D):
            rho[j * D + j, k * D + k] = np.sqrt(pops[j] * pops[k]) * c ** abs(j - k)
    coherent = qt.Qobj(rho, dims=[[D, D], [D, D]])
    return p * coherent + (1 - p) * qt.tensor(qt.qeye(D), qt.qeye(D)) / D**2

def fidelity(D, **kw):
    return qt.expect(qt.ket2dm(ideal(D)), state(D, **kw)).real

## 1. Checks of the model against closed-form fidelities

In [3]:
check("ideal state", fidelity(3, p=1.0), 1.0)
check("white noise: p + (1-p)/D^2", fidelity(4, p=0.9), 0.9 + 0.1 / 16)
pops = 0.5 ** np.arange(4) / sum(0.5 ** np.arange(4))
check("unequal bins: (sum sqrt p_k)^2 / D", fidelity(4, p=1, r=0.5), np.sum(np.sqrt(pops)) ** 2 / 4)

PASS  ideal state: simulation = 1.0000, theory = 1.0000
PASS  white noise: p + (1-p)/D^2: simulation = 0.9063, theory = 0.9062
PASS  unequal bins: (sum sqrt p_k)^2 / D: simulation = 0.8743, theory = 0.8743


## 2. Three hypotheses, each fitted to the three fidelities

| Hypothesis | Physical picture | Free parameters |
|---|---|---|
| H1 white noise | accidental coincidences, equal in every dimension | p |
| H2 + unequal bins | some frequency pairs brighter than others | p, r |
| H3 + dephasing | phase stability between bins degrades with their separation | p, c |

**Gap:** two parameters fitted to three points can fit many curves. A good fit makes a hypothesis possible, not proven.

In [4]:
def fit(model, x0, bounds):
    res = least_squares(lambda x: [model(D, x) - F_MEAS[D] for D in DIMS], x0, bounds=bounds)
    return res.x, [model(D, res.x) for D in DIMS]

hyps = {
    "H1 white noise":       (lambda D, x: fidelity(D, p=x[0]), [0.9], ([0], [1])),
    "H2 + unequal bins":    (lambda D, x: fidelity(D, p=x[0], r=x[1]), [0.9, 0.8], ([0, 0.05], [1, 1])),
    "H3 + dephasing":       (lambda D, x: fidelity(D, p=x[0], c=x[1]), [0.9, 0.9], ([0, 0], [1, 1])),
}
fitted = {}
print(f"{'hypothesis':<20}{'parameters':<22}{'fidelities D=2,3,4':<26}worst miss")
print(f"{'measured':<20}{'':<22}{str(list(F_MEAS.values())):<26}")
for name, (model, x0, bounds) in hyps.items():
    params, pred = fit(model, x0, bounds)
    fitted[name] = params
    worst = max(abs(pr - F_MEAS[D]) for pr, D in zip(pred, DIMS))
    print(f"{name:<20}{str([round(float(v), 3) for v in params]):<22}{str([round(float(v), 3) for v in pred]):<26}{worst:.3f}")

hypothesis          parameters            fidelities D=2,3,4        worst miss
measured                                  [0.885, 0.809, 0.766]     
H1 white noise      [0.787]               [0.84, 0.811, 0.8]        0.045
H2 + unequal bins   [0.876, 0.477]        [0.879, 0.819, 0.761]     0.010
H3 + dephasing      [0.957, 0.819]        [0.882, 0.815, 0.763]     0.006


H1 fails: with fixed noise, $F = p + (1-p)/D^2$ is almost flat in D, but the data fall. H2 and H3 both fit. They need very different states:

In [5]:
D = 4
p2, r2 = fitted["H2 + unequal bins"]
p3, c3 = fitted["H3 + dephasing"]
for name, kw in [("H2 unequal bins", dict(p=p2, r=r2)), ("H3 dephasing", dict(p=p3, c=c3))]:
    rho = state(D, **kw).full()
    diag = [rho[k * D + k, k * D + k].real for k in range(D)]
    near = abs(rho[0, 1 * D + 1]) / np.sqrt(diag[0] * diag[1])
    far = abs(rho[0, 3 * D + 3]) / np.sqrt(diag[0] * diag[3])
    print(f"{name:<16} |kk> populations {np.round(diag, 3)}   coherence near / far {near:.2f} / {far:.2f}")

H2 unequal bins  |kk> populations [0.491 0.238 0.118 0.06 ]   coherence near / far 0.98 / 0.93
H3 dephasing     |kk> populations [0.242 0.242 0.242 0.242]   coherence near / far 0.81 / 0.54


## 3. Verdict

| Hypothesis | Evidence | Status |
|---|---|---|
| H1 fixed white noise | cannot reproduce the fall with D (worst miss 0.045) | ruled out |
| H2 unequal bins | needs an 8× brightness spread across 4 bins; the measured joint spectral intensity (Fig. 2a) shows all 10 bin pairs within about 2× of each other; the ring's own dispersion gives a bin-imbalance fidelity above 0.99999 for the nearest 4 bins (notebook 13) | ruled out |
| H3 dephasing growing with bin separation | fits; would show as off-diagonal elements shrinking with distance from the diagonal | open |
| H4 noise growing with D | D = 4 mixes with 4 % sideband efficiency instead of 11 %, so fewer true coincidences against the same accidentals; consistent with the authors' statement that loss and CAR limit fidelity | open, favoured by the setup details |

**Measurement that separates H3 from H4:** two-photon interference between bin pairs at increasing separation (1, 2, 3 FSR) at a fixed modulation efficiency. H3 predicts visibility falling with separation; H4 predicts no dependence on separation once the efficiency is fixed.

**Conclusion:** the fall of fidelity with dimension is most likely caused by the measurement setup (accidentals and possibly phase noise between bins), not by the photon source on the chip.

## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [6]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

3 of 3 checks passed
